# OMST filtering of averaged Pearson correlation matrices

Apply OMST (Orthogonal Minimum Spanning Trees, global-cost-efficiency)
filtering to every matrix produced by `average_corr.ipynb`. Three sign
strategies are saved per input matrix:

* **`A_pos`** — zero negatives, OMST on positive subgraph (output ≥ 0)
* **`A_neg`** — zero positives, abs, OMST, negate output (output ≤ 0)
* **`B`**     — abs, OMST, restore original signs (signed output)

Inputs are read from `pearson_avg_matrices/<sub>/<stem>.npy` (4 sub-folders,
180 files total). Outputs are written to a parallel folder
`pearson_avg_matrices_omst/<sub>/<stem>__<strategy>.{npy,csv}` plus per-folder
and master metrics CSVs under `pearson_avg_matrices_omst/metrics/`.

Filtering is done by `omst_filter.omst_filter_batch`, which delegates to the
MATLAB function `threshold_omst_gce_wu_very_fast` in
`/home/aazarg/data/topological_filtering_networks/`. One MATLAB subprocess
per sub-folder (4 calls total).

In [2]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from omst_filter import omst_filter_batch, ALL_STRATEGIES

ROOT = Path('/home/aazarg/data/rchimera_plots')
IN_BASE  = ROOT / 'pearson_avg_matrices'
OUT_BASE = ROOT / 'pearson_avg_matrices_omst'
MET_DIR  = OUT_BASE / 'metrics'
MET_DIR.mkdir(parents=True, exist_ok=True)

SUBFOLDERS = [
    'group_at_each_timepoint_average',
    'pre_and_post_comb_average',
    'sex_diff_group_at_each_timepoint_average',
    'sex_diff_pre_and_post_comb_average',
]
STRATEGIES = list(ALL_STRATEGIES)
print('Strategies:', STRATEGIES)

In [2]:
# Parse the file-name conventions used by average_corr.ipynb back into the
# (group, sex, day, phase) tuple. Sex is 'ALL' for the non-sex-split folders;
# phase is 'COMB' for the pre/post-combined folders.
GROUP_TOKENS = ['Shm_rCHIMERA', 'Shm_rCHI', 'rCHIMERA', 'rCHI']  # longest first

def parse_stem(stem: str, subfolder: str) -> dict:
    for grp in GROUP_TOKENS:
        if stem.startswith(grp + '_'):
            rest = stem[len(grp) + 1:]
            break
    else:
        raise ValueError(f'Cannot parse group from stem {stem!r}')

    parts = rest.split('_')
    sex = 'ALL'
    if 'sex_diff' in subfolder:
        sex = parts.pop(0)  # 'M' or 'F'
    day = int(parts[0].lstrip('D'))
    phase = parts[1] if len(parts) > 1 else 'COMB'
    return {'stem': stem, 'group': grp.replace('_', ' '), 'sex': sex,
            'day': day, 'phase': phase}

# quick sanity check
for s, sub in [
    ('rCHI_D1', 'pre_and_post_comb_average'),
    ('Shm_rCHIMERA_D3_POST', 'group_at_each_timepoint_average'),
    ('Shm_rCHI_F_D2', 'sex_diff_pre_and_post_comb_average'),
    ('rCHIMERA_M_D1_PRE', 'sex_diff_group_at_each_timepoint_average'),
]:
    print(parse_stem(s, sub))

{'stem': 'rCHI_D1', 'group': 'rCHI', 'sex': 'ALL', 'day': 1, 'phase': 'COMB'}
{'stem': 'Shm_rCHIMERA_D3_POST', 'group': 'Shm rCHIMERA', 'sex': 'ALL', 'day': 3, 'phase': 'POST'}
{'stem': 'Shm_rCHI_F_D2', 'group': 'Shm rCHI', 'sex': 'F', 'day': 2, 'phase': 'COMB'}
{'stem': 'rCHIMERA_M_D1_PRE', 'group': 'rCHIMERA', 'sex': 'M', 'day': 1, 'phase': 'PRE'}


In [3]:
def process_subfolder(subfolder: str) -> pd.DataFrame:
    """Run OMST on every .npy in IN_BASE/subfolder; save filtered npy+csv
    and return a DataFrame of metrics for each (file, strategy) pair."""
    in_dir  = IN_BASE  / subfolder
    out_dir = OUT_BASE / subfolder
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob('*.npy'))
    if not files:
        print(f'No .npy files in {in_dir}')
        return pd.DataFrame()

    # Stack into (K, N, N) for one batched MATLAB call.
    stems = [f.stem for f in files]
    stack = np.stack([np.load(f) for f in files], axis=0).astype(np.float64)
    print(f'\n=== {subfolder}: K={stack.shape[0]} matrices ===')

    filtered, metrics = omst_filter_batch(stack, strategies=STRATEGIES)

    rows = []
    for k, stem in enumerate(stems):
        meta = parse_stem(stem, subfolder)
        for strat in STRATEGIES:
            mat = filtered[strat][k]
            np.save(out_dir / f'{stem}__{strat}.npy', mat)
            np.savetxt(out_dir / f'{stem}__{strat}.csv', mat,
                       delimiter=',', fmt='%.8f')
            rows.append({
                **meta,
                'strategy': strat,
                'subfolder': subfolder,
                **{f: metrics[strat][k][f]
                   for f in ('n_msts', 'mdeg', 'gce', 'costmax', 'E')},
            })

    df = pd.DataFrame(rows)
    csv_path = MET_DIR / f'{subfolder}__metrics.csv'
    df.to_csv(csv_path, index=False)
    print(f'  wrote {len(rows)} filtered matrices ({len(stems)} inputs x '
          f'{len(STRATEGIES)} strategies); metrics -> {csv_path.name}')
    return df

In [4]:
all_dfs = []
for sub in SUBFOLDERS:
    df = process_subfolder(sub)
    if not df.empty:
        all_dfs.append(df)

master = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
master_path = MET_DIR / 'all_metrics.csv'
master.to_csv(master_path, index=False)
print(f'\nMaster metrics: {len(master)} rows -> {master_path}')
print(master.head())


=== group_at_each_timepoint_average: K=40 matrices ===
[omst] 40 matrices x 3 strategies
OMST batch: K=40 matrices, N=106 nodes, strategies=A_pos,A_neg,B
  Strategy A_pos ...
  Strategy A_neg ...
  Strategy B ...
Saving /tmp/omst_ok61eijm/omst_output.mat ...
Done.
  wrote 120 filtered matrices (40 inputs x 3 strategies); metrics -> group_at_each_timepoint_average__metrics.csv

=== pre_and_post_comb_average: K=20 matrices ===
[omst] 20 matrices x 3 strategies
OMST batch: K=20 matrices, N=106 nodes, strategies=A_pos,A_neg,B
  Strategy A_pos ...
  Strategy A_neg ...
  Strategy B ...
Saving /tmp/omst_yb2f2y4_/omst_output.mat ...
Done.
  wrote 60 filtered matrices (20 inputs x 3 strategies); metrics -> pre_and_post_comb_average__metrics.csv

=== sex_diff_group_at_each_timepoint_average: K=80 matrices ===
[omst] 80 matrices x 3 strategies
OMST batch: K=80 matrices, N=106 nodes, strategies=A_pos,A_neg,B
  Strategy A_pos ...
  Strategy A_neg ...
  Strategy B ...
Saving /tmp/omst_h63glbp1/omst

In [5]:
# File counts and sanity stats per folder/strategy.
print('Output file counts:')
for sub in SUBFOLDERS:
    d = OUT_BASE / sub
    n_npy = len(list(d.glob('*.npy')))
    n_csv = len(list(d.glob('*.csv')))
    print(f'  {sub:<45s} {n_npy:4d} .npy   {n_csv:4d} .csv')

if not master.empty:
    print('\nMean metrics by strategy:')
    print(master.groupby('strategy')[['n_msts','mdeg','gce','costmax','E']]
          .mean().round(4))
    print('\nNon-zero edges per filtered matrix (sanity check):')
    nnz_rows = []
    for sub in SUBFOLDERS:
        for f in (OUT_BASE / sub).glob('*.npy'):
            arr = np.load(f)
            strat = f.stem.split('__')[-1]
            nnz_rows.append({'subfolder': sub, 'strategy': strat,
                             'nnz': int(np.count_nonzero(arr))})
    nnz_df = pd.DataFrame(nnz_rows)
    print(nnz_df.groupby(['subfolder','strategy'])['nnz']
          .agg(['mean','min','max']).round(1))

Output file counts:
  group_at_each_timepoint_average                120 .npy    120 .csv
  pre_and_post_comb_average                       60 .npy     60 .csv
  sex_diff_group_at_each_timepoint_average       240 .npy    240 .csv
  sex_diff_pre_and_post_comb_average             120 .npy    120 .csv

Mean metrics by strategy:
          n_msts     mdeg     gce  costmax       E
strategy                                          
A_neg     2.0000   1.9811  0.4882   0.0240  0.0000
A_pos     8.4333  14.7264  0.4712   0.1734  0.3479
B         8.4333  14.7264  0.4712   0.1734  0.3479

Non-zero edges per filtered matrix (sanity check):
                                                     mean   min   max
subfolder                                strategy                    
group_at_each_timepoint_average          A_neg      210.0   210   210
                                         A_pos     1596.0  1260  1890
                                         B         1596.0  1260  1890
pre_and_post_com